<a href="https://colab.research.google.com/github/DarkLyng/Proyecto_Modelos/blob/main/Proyecto_Modelos_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. Preparar datos**

In [133]:
# Bibliotecas de Python

import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import csv

warnings.filterwarnings('ignore')

In [134]:
# Carga el archivo xlsx
xls = pd.ExcelFile("/content/Contoso Sales.xlsx")
print(xls.sheet_names)

['DimChannel', 'DimGeography', 'DimProduct', 'DimProductCategory', 'DimProductSubCategory', 'DimPromotion', 'DimStores']


In [135]:
# Crea una lista con los nombres de las hojas de Contoso Sales
tabs = ['DimChannel', 'DimGeography', 'DimProduct', 'DimProductCategory', 'DimProductSubCategory', 'DimPromotion', 'DimStores']
tablas = []

In [136]:
# Se agrega en la lista los nombres de las hojas del xlsx
for tab in tabs:
  df = pd.read_excel("/content/Contoso Sales.xlsx", sheet_name=tab)
  tablas.append(df)

In [137]:
# Lee el el archivo FactSales, lo separa y agrega a la lista "tablas" los datos del txt
ventas = pd.read_csv("FactSales.txt", sep="\t")
tablas.append(ventas)

In [138]:
# Muestra la cantidad de tablas que se cargaron a la lista
print(f"Total de tablas cargadas en la lista: {len(tablas)}")

Total de tablas cargadas en la lista: 8


# **2. Transformación**

Cambiar dtype de ciertas columnas

In [180]:
df_ventas = tablas[-1]  # Último índice
df_stores = tablas[6]   # Índice 0: Tiendas
df_product= tablas[2]   # Índice 2: Productos

In [140]:
df_ventas['DateKey'] = pd.to_datetime(df_ventas['DateKey'], errors='coerce')

In [141]:
df_ventas['ReturnQuantity'] = pd.to_numeric(df_ventas['ReturnQuantity'], errors='coerce').fillna(0).astype(int)

In [142]:
df_ventas['Cantidad_total'] = pd.to_numeric(df_ventas['Cantidad_total'], errors='coerce').fillna(0).astype(int)

In [143]:
df_ventas['Devolución'] = df_ventas['Devolución'].astype(str)

In [144]:
df_ventas['Venta_type'] = df_ventas['Venta_type'].astype(str)

In [145]:
df_ventas['Devolución_type'] = df_ventas['Devolución_type'].astype(str)

In [146]:
df_ventas['Precio_unitario'] = pd.to_numeric(df_ventas['Precio_unitario'], errors='coerce').fillna(0).astype(float)

In [147]:
df_ventas['Ingresos'] = pd.to_numeric(df_ventas['Ingresos'], errors='coerce').fillna(0).astype(float)

Métricas de las Tablas FactSales

In [161]:
cantidad_ventas = df_ventas['SalesQuantity'].sum()
print(f" Cantidad total de ventas (Unidades): {cantidad_ventas:.0f}")

 Cantidad total de ventas (Unidades): 37007006


In [162]:
cantidad_devoluciones = df_ventas['ReturnQuantity'].sum()
print(f" Cantidad total de devoluciones (Unidades): {cantidad_devoluciones:.0f}")

 Cantidad total de devoluciones (Unidades): 337999


In [163]:
cantidad_total = cantidad_ventas - cantidad_devoluciones
print(f" Cantidad total de ventas menos devoluciones (Cantidad Total): {cantidad_total:.0f}")

 Cantidad total de ventas menos devoluciones (Cantidad Total): 36669007


Otras Métricas

In [167]:
promedio_precio_unitario = df_product['UnitPrice'].mean()
print(f" Promedio precio unitario: ₡{promedio_precio_unitario:.2f}")

 Promedio Precio Unitario: ₡316.92


In [181]:
numero_stores = df_stores['StoreKey'].count()
print(f" Número total de tiendas registradas: {numero_stores:}")

 Número total de tiendas registradas: 310


In [182]:
numero_stores_con_ventas = df_ventas['StoreKey'].nunique()
print(f" Número de tiendas con ventas reales: {numero_stores_con_ventas:,}")

 Número de tiendas con ventas reales: 306


In [176]:
numero_ordenes = len(df_ventas)
print(f" Número total de ordenes procesadas: {numero_ordenes:}")

 Número total de ordenes procesadas: 2294174


In [177]:
numero_ordenes_con_devolucion = len(df_ventas[df_ventas['ReturnQuantity'] > 0])
print(f" Órdenes con devolución: {numero_ordenes_con_devolucion:}")

 Órdenes con devolución: 330241


In [178]:
total_ingresos = df_ventas['Ingresos'].sum()
print(f" Total de ingresos históricos: ₡{total_ingresos:.2f}")

 Total de ingresos históricos: ₡129702883.00


# **3. Carga**

In [152]:
# Comproba por separado el número de columnas y filas del dataframe
tablas[2].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1690 entries, 0 to 1689
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ProductKey             1690 non-null   int64  
 1   ProductName            1690 non-null   object 
 2   ProductDescription     1690 non-null   object 
 3   ProductSubcategoryKey  1690 non-null   int64  
 4   Manufacturer           1690 non-null   object 
 5   BrandName              1690 non-null   object 
 6   ClassID                1690 non-null   int64  
 7   ClassName              1690 non-null   object 
 8   ColorID                1690 non-null   int64  
 9   ColorName              1690 non-null   object 
 10  Size                   1240 non-null   object 
 11  Weight                 1526 non-null   float64
 12  UnitCost               1690 non-null   float64
 13  UnitPrice              1690 non-null   float64
dtypes: float64(3), int64(4), object(7)
memory usage: 185.0+ 

In [153]:
# Muestra la cantidad de filas columnas y datos faltantes del dataframe
print(" Conteo de filas, columnas y datos fantantes \n")

for i, df in enumerate(tablas):
    # Filas y columnas
    filas = df.shape[0]
    columnas = df.shape[1]
    # Total de celdas vacías o con errores en toda la tabla
    celdas_vacias_tot = df.isnull().sum().sum()

    # Nombre original usando la lista tabs
    nombre_tabla = tabs[i] if i < len(tabs) else "FactSales"
    print(f"Índice [{i}] - Tabla: '{nombre_tabla}'")
    print(f" Filas: {filas}")
    print(f" Columnas: {columnas}")
    print(f" Total de celdas vacías o errores encontrados: {celdas_vacias_tot}")
    print("-" * 40)

 Conteo de filas, columnas y datos fantantes 

Índice [0] - Tabla: 'DimChannel'
 Filas: 4
 Columnas: 2
 Total de celdas vacías o errores encontrados: 0
----------------------------------------
Índice [1] - Tabla: 'DimGeography'
 Filas: 674
 Columnas: 3
 Total de celdas vacías o errores encontrados: 3
----------------------------------------
Índice [2] - Tabla: 'DimProduct'
 Filas: 1690
 Columnas: 14
 Total de celdas vacías o errores encontrados: 614
----------------------------------------
Índice [3] - Tabla: 'DimProductCategory'
 Filas: 8
 Columnas: 2
 Total de celdas vacías o errores encontrados: 0
----------------------------------------
Índice [4] - Tabla: 'DimProductSubCategory'
 Filas: 44
 Columnas: 3
 Total de celdas vacías o errores encontrados: 0
----------------------------------------
Índice [5] - Tabla: 'DimPromotion'
 Filas: 28
 Columnas: 7
 Total de celdas vacías o errores encontrados: 0
----------------------------------------
Índice [6] - Tabla: 'DimStores'
 Filas: 310


In [154]:
# Función que permite obtener un análisis estadístico de cada hoja de Contoso Sales

def analizar_tabla_lista(df, nombre_tabla):
    #print("\n" + "="*40)
    print(f"Análisis de la tabla: {nombre_tabla}")
    #print("="*60)

    for col in df.columns:
        dtype = df[col].dtype
        total_valores = len(df[col])
        valores_unicos = df[col].nunique()
        faltantes = df.isnull().sum().sum()

        print(f"\n Columna: '{col}' | Tipo: {dtype}")
        print(f"  - Datos faltantes: {faltantes} ({ (faltantes/total_valores)*100:.2f}%)")
        print(f"  - Cardinalidad (Valores únicos): {valores_unicos}")


        # Variables numéricas continuas o cuantitativas
        if np.issubdtype(dtype, np.number) and valores_unicos > 10:
            print("  - Clasificación: Variable numérica continua")

            media = df[col].mean()
            q1 = df[col].quantile(0.25)   # Primer Cuartil (25%)
            mediana = df[col].median()
            q3 = df[col].quantile(0.75)   # Tercer Cuartil (75%)
            desv_est = df[col].std()
            varianza = df[col].var()
            inclinacion = df[col].skew()  # Inclinación(Skewness)
            curtosis = df[col].kurt()     # Kurtosis
            val_min = df[col].min()
            val_max = df[col].max()
            # Cálculo de los outliers
            ric = q3 - q1
            limite_inferior_outlier = q1 - 1.5 * ric
            limite_superior_outlier = q3 + 1.5 * ric

            print(f"  - Valor esperado (Media): {media:.4f}")
            print(f"  - Cuartil 1 (Q1 - 25%): {q1:.4f}")
            print(f"  - Mediana: {mediana:.4f}")
            print(f"  - Cuartil 3 (Q3 - 75%): {q3:.4f}")
            print(f"  - Varianza: {varianza:.4f}")
            print(f"  - Desviación estándar: {desv_est:.4f}")
            print(f"  - Inclinación (Skewness): {inclinacion:.4f}")
            print(f"  - Kurtosis: {curtosis:.4f}")

            if abs(inclinacion) < 0.5:
                  print("  -  Distribución simétrica.")
            else:
                print(f"  - Distribución asimétrica a la {'derecha' if inclinacion > 0 else 'izquierda'}.")

            print(f"  - Valor mínimo: {val_min:.4f}")
            print(f"  - Valor máximo: {val_max:.4f}")
            print(f"  - Rango intercuartílico (RIC): {ric:.4f}")
            print(f"  - Límites para outliers: [{limite_inferior_outlier:.2f} , {limite_superior_outlier:.2f}]")

        # Variables discretas o cualitativas (Frecuencia y probabilidad empírica)
        else:
            print("  - Clasificación: Variable cualitativa o discreta")
            frecuencias = df[col].value_counts(dropna=False).head(5)
            probabilidades = df[col].value_counts(dropna=False, normalize=True).head(5)

            print("  - Los 5 valores con mayor frecuencia y probabilidad empírica P(X=x):")
            for val, freq in frecuencias.items():
                prob = probabilidades[val]
                print(f"     Valor: {val} | Frecuencia: {freq} | Probabilidad: {prob:.4f} ({prob*100:.2f}%)")



In [155]:
# Se analiza por índices
# Analiza la tabla DimProduct
analizar_tabla_lista(tablas[2], "DimProduct")

Análisis de la tabla: DimProduct

 Columna: 'ProductKey' | Tipo: int64
  - Datos faltantes: 614 (36.33%)
  - Cardinalidad (Valores únicos): 1690
  - Clasificación: Variable numérica continua
  - Valor esperado (Media): 860.1805
  - Cuartil 1 (Q1 - 25%): 423.2500
  - Mediana: 845.5000
  - Cuartil 3 (Q3 - 75%): 1267.7500
  - Varianza: 274465.4766
  - Desviación estándar: 523.8945
  - Inclinación (Skewness): 0.4078
  - Kurtosis: -0.0250
  -  Distribución simétrica.
  - Valor mínimo: 1.0000
  - Valor máximo: 2517.0000
  - Rango intercuartílico (RIC): 844.5000
  - Límites para outliers: [-843.50 , 2534.50]

 Columna: 'ProductName' | Tipo: object
  - Datos faltantes: 614 (36.33%)
  - Cardinalidad (Valores únicos): 1690
  - Clasificación: Variable cualitativa o discreta
  - Los 5 valores con mayor frecuencia y probabilidad empírica P(X=x):
     Valor: Contoso In-Line Coupler E180 Black | Frecuencia: 1 | Probabilidad: 0.0006 (0.06%)
     Valor: Contoso Wireless Laser Mouse E50 Grey | Frecuenci

In [156]:
# Analiza la tabla FactSales (Último índice de la lista)
# analizar_tabla_lista(tablas[-1], "FactSales")

In [157]:
# Forma fácil de obtener los valores atípicos, limites, cuartiles, etc
#print(tablas[7]["SalesQuantity"].describe())

In [158]:
#tablas[7]["SalesQuantity"].plot.kde()
#plt.xlim([0,50])

In [159]:
#tablas[6]["GeographyKey"].hist()

In [ ]:
#print(tablas[6]["GeographyKey"].head())

In [160]:

        # --- CREACIÓN AUTOMÁTICA DE GRÁFICOS COMPUESTOS (Histograma + Boxplot) ---
            # Creamos una figura con dos subgráficos: arriba el histograma y abajo el diagrama de caja
            fig, (ax_box, ax_hist) = plt.subplots(2, 1, sharex=True, gridspec_kw={"height_ratios": (.20, .80)}, figsize=(9, 5))

            # Control de Outliers visuales específico para FactSales (debido al máximo extremo de 1920)
            if nombre_tabla == "FactSales" and col in ['SalesQuantity', 'ReturnQuantity']:
                limite_visual = df[col].quantile(0.99)
                datos_graficar = df[df[col] <= limite_visual][col]
                fig.suptitle(f"Análisis Probabilístico de {col} en {nombre_tabla} (Filtro Visual <={limite_visual})", fontsize=12, fontweight='bold')
            else:
                datos_graficar = df[col].dropna()
                fig.suptitle(f"Análisis Probabilístico y de Posición: {col} ({nombre_tabla})", fontsize=12, fontweight='bold')

            # 1. Gráfico de Caja (Arriba) - Muestra visualmente Min, Q1, Q2, Q3, Max y Outliers
            sns.boxplot(x=datos_graficar, ax=ax_box, color='lightgreen', fliersize=4)
            ax_box.set(xlabel='') # Quitamos el eje X del boxplot para no duplicar
            ax_box.set_title(f"Mín: {datos_graficar.min():.1f} | Q1: {q1:.1f} | Q2: {q2:.1f} | Q3: {q3:.1f} | Máx: {datos_graficar.max():.1f}", fontsize=9, style='italic')

            # 2. Gráfico de Histograma + Densidad KDE (Abajo)
            sns.histplot(datos_graficar, kde=True, color='royalblue', stat="density", bins=30, ax=ax_hist)
            ax_hist.axvline(media, color='red', linestyle='--', linewidth=1.5, label=f'Media ({media:.2f})')
            ax_hist.axvline(mediana, color='green', linestyle='-', linewidth=1.5, label=f'Mediana ({mediana:.2f})')

            ax_hist.set_xlabel(col)
            ax_hist.set_ylabel("Densidad de Probabilidad")
            ax_hist.legend()

            plt.tight_layout()
            plt.show()

IndentationError: unexpected indent (2528388690.py, line 3)